In [17]:
from matplotlib.pyplot import stem
from pandas.core.arrays import categorical
from pyparsing import alphas
%pip install -q Pandas
%pip install -q pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

* Dataset source

In [19]:
df = pd.read_csv('../Data/Zillow_top20_states.csv')
df.head()

,zpid,state_code,state_name,address,street,city,zipcode,price,price_formatted,beds,...,latitude,longitude,status,home_type,days_on_zillow,zestimate,detail_url,has_open_house,open_house_date,is_featured
0,17264897,CA,California,"979 Kevin Ave, Redlands, CA 92373",979 Kevin Ave,Redlands,92373,447000,"$447,000",3.0,...,34.040520,-117.186195,House for sale,SINGLE_FAMILY,5,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,NaN,False
1,20021372,CA,California,"13114 Addison St, Sherman Oaks, CA 91423",13114 Addison St,Sherman Oaks,91423,2795000,"$2,795,000",4.0,...,34.160885,-118.418770,House for sale,SINGLE_FAMILY,8,NaN,https://www.zillow.com/homedetails/13114-Addis...,True,2026-02-08T12:00:00,False
2,20009320,CA,California,"6032 Goodland Ave, North Hollywood, CA 91606",6032 Goodland Ave,North Hollywood,91606,1718000,"$1,718,000",4.0,...,34.180450,-118.411280,House for sale,SINGLE_FAMILY,8,1695500.0,https://www.zillow.com/homedetails/6032-Goodla...,False,NaN,False
3,460204882,CA,California,"3325 Tonopah St, Oceanside, CA 92054",3325 Tonopah St,Oceanside,92054,899999,"$899,999",3.0,...,33.214260,-117.341500,Coming soon,SINGLE_FAMILY,4,NaN,https://www.zillow.com/homedetails/3325-Tonopa...,False,NaN,False
4,20769150,CA,California,"963 Pine Grove Ave, Los Angeles, CA 90042",963 Pine Grove Ave,Los Angeles,90042,995000,"$995,000",2.0,...,34.132890,-118.186220,House for sale,SINGLE_FAMILY,5,999900.0,https://www.zillow.com/homedetails/963-Pine-Gr...,False,NaN,False


In [20]:
df.isnull().sum().sort_values(ascending=False)
df.info()
df.describe()
print(df.columns)

<class 'pandas.DataFrame'>
RangeIndex: 2214 entries, 0 to 2213
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   zpid             2214 non-null   int64  
 1   state_code       2214 non-null   str    
 2   state_name       2214 non-null   str    
 3   address          2214 non-null   str    
 4   street           2214 non-null   str    
 5   city             2214 non-null   str    
 6   zipcode          2214 non-null   int64  
 7   price            2214 non-null   int64  
 8   price_formatted  2214 non-null   str    
 9   beds             2205 non-null   float64
 10  baths            2206 non-null   float64
 11  area_sqft        2154 non-null   float64
 12  latitude         2210 non-null   float64
 13  longitude        2210 non-null   float64
 14  status           2214 non-null   str    
 15  home_type        2214 non-null   str    
 16  days_on_zillow   2214 non-null   int64  
 17  zestimate        1102 non

In [21]:
df = df.dropna(subset=["price"])

df = df.fillna(df.median(numeric_only=True))

colsToDrop = ["zpid", "address", "street", "price_formatted", "detail_url"]
df.drop(columns=colsToDrop, errors="ignore", inplace=True)

df.shape

(2214, 17)

In [22]:
df.to_csv('../Data/cleaned.csv', index=False)
df = pd.read_csv('../Data/cleaned.csv')

In [23]:
X = df.drop(columns=["price"])
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

((1771, 16), (443, 16))

In [24]:
# Predict using mean of training price
mean_price = y_train.mean()
y_pred_mean = np.full(shape=len(y_test), fill_value=mean_price)

mae_mean = mean_absolute_error(y_test, y_pred_mean)
mse_mean = mean_squared_error(y_test, y_pred_mean)
r2_mean = r2_score(y_test, y_pred_mean)

mae_mean, mse_mean, r2_mean

(1565788.9099028364, 63591937013662.77, -1.4559892440590971e-06)

In [25]:
# Use a single feature
X_train_area = X_train[["area_sqft"]]
X_test_area = X_test[["area_sqft"]]

lin_simple = LinearRegression()
lin_simple.fit(X_train_area, y_train)

y_pred_simple = lin_simple.predict(X_test_area)

mae_simple = mean_absolute_error(y_test, y_pred_simple)
mse_simple = mean_squared_error(y_test, y_pred_simple)
r2_simple = r2_score(y_test, y_pred_simple)

mae_simple, mse_simple, r2_simple

(1867118.9336945068, 44194016966927.49, 0.3050364025954151)

In [26]:
# Compare
baseline_result = pd.DataFrame({
    "Model": ["Mean Predict", "Simple Linear Regression"],
    "MAE": [mae_mean, mae_simple],
    "MSE": [mse_mean, mse_simple],
    "R2": [r2_mean, r2_simple],
})
baseline_result

,Model,MAE,MSE,R2
0,Mean Predict,1.565789e+06,6.359194e+13,-0.000001
1,Simple Linear Regression,1.867119e+06,4.419402e+13,0.305036


In [28]:
# Separate features by type
numeric_features = df.select_dtypes(include=["int64", "float64"]).drop(columns=["price"]).columns
categorical_features = df.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransform(
    transformers = [
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

C:\Users\sagga\AppData\Local\Temp\ipykernel_19828\1354404397.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include=["object"]).columns


In [29]:
lin_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("Model", LinearRegression())
])

lin_reg.fit(X_train, y_train)
y_pred_lr = lin_reg.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

mae_lr, mse_lr, r2_lr

(2263020.9711624994, 152349125127566.28, -1.395733706201172)

In [30]:
ridge = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("Model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

mae_ridge, mse_ridge, r2_ridge

(2324460.7234944482, 52628148952434.5, 0.17240725711584937)

In [31]:
lasso = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("Model", Lasso(alpha=0.001)),
])

lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

mae_lasso, mse_lasso, r2_lasso

C:\Users\sagga\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.538e+15, tolerance: 1.386e+13
  model = cd_fast.sparse_enet_coordinate_descent(


(2211098.7163179917, 151557524855522.72, -1.3832855647892357)

In [32]:
sgd = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("Model", SGDRegressor(max_iter=1000, tol=0.001, random_state=42))
])

sgd.fit(X_train, y_train)
y_pred_sgd = sgd.predict(X_test)

mae_sgd = mean_absolute_error(y_test, y_pred_sgd)
mse_sgd = mean_squared_error(y_test, y_pred_sgd)
r2_sgd = r2_score(y_test, y_pred_sgd)

mae_sgd, mse_sgd, r2_sgd

(2317933.231243642, 41178366960511.71, 0.3524583642274667)

In [33]:
# Preprocessing. Separate features by type
numeric_features = [
    "beds",
    "baths",
    "area_sqft",
    "latitude",
    "longitude",
    "days_on_zillow"
]

Categorical_features = [
    "city"
]

X = df[numeric_features]
y = df[["price"]]
X.head()

,beds,baths,area_sqft,latitude,longitude,days_on_zillow
0,3.0,2.0,1300.0,34.040520,-117.186195,5
1,4.0,5.0,2900.0,34.160885,-118.418770,8
2,4.0,3.0,3025.0,34.180450,-118.411280,8
3,3.0,3.0,1682.0,33.214260,-117.341500,4
4,2.0,2.0,1271.0,34.132890,-118.186220,5


In [34]:
# re-running the train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

mae_lr, mse_lr, r2_lr

(2225316.292212382, 42620546487641.9, 0.3297796773584347)

In [37]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

mae_ridge, mse_ridge, r2_ridge

(2225196.897338073, 42624318726279.46, 0.32972035782348974)

In [38]:
# Lasso Regression
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

mae_lasso, mse_lasso, r2_lasso

(2225316.2918753754, 42620546488003.03, 0.3297796773527557)

In [39]:
# SGDRegressor
sgd = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(X_train_scaled, y_train.values.ravel())

y_pred_sgd = sgd.predict(X_test_scaled)

mae_sgd = mean_absolute_error(y_test, y_pred_sgd)
mse_sgd = mean_squared_error(y_test, y_pred_sgd)
r2_sgd = r2_score(y_test, y_pred_sgd)

mae_sgd, mse_sgd, r2_sgd

(2290148.4139093584, 42830170630786.33, 0.32648327755998396)

In [40]:
# Comparison table
final_results = pd.DataFrame({
    "Model": [
        "Mean Predictor",
        "Simple Linear Regression (Area Only)",
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "SGD Regressor"
    ],
    "MAE": [
        mae_mean,
        mae_simple,
        mae_lr,
        mae_ridge,
        mae_lasso,
        mae_sgd
    ],
    "MSE": [
        mse_mean,
        mse_simple,
        mse_lr,
        mse_ridge,
        mse_lasso,
        mse_sgd
    ],
    "R2": [
        r2_mean,
        r2_simple,
        r2_lr,
        r2_ridge,
        r2_lasso,
        r2_sgd
    ]
})

final_results

,Model,MAE,MSE,R2
0,Mean Predictor,1.565789e+06,6.359194e+13,-0.000001
1,Simple Linear Regression (Area Only),1.867119e+06,4.419402e+13,0.305036
2,Linear Regression,2.225316e+06,4.262055e+13,0.329780
3,Ridge Regression,2.225197e+06,4.262432e+13,0.329720
4,Lasso Regression,2.225316e+06,4.262055e+13,0.329780
5,SGD Regressor,2.290148e+06,4.283017e+13,0.326483
